💊 CNN Klasifikasi Jenis Obat Tablet
**Dataset:** OGYEIv2 — 112 kelas obat tablet (Kaggle)

**Link Dataset:** https://www.kaggle.com/datasets/richardradli/ogyeiv2


In [ ]:
import tensorflow as tf

print('TensorFlow version:', tf.__version__)
gpu = tf.config.list_physical_devices('GPU')
if gpu:
    print('✅ GPU aktif:', gpu[0])
else:
    print('⚠️ GPU tidak aktif! Aktifkan dulu via Runtime → Change runtime type')

TensorFlow version: 2.20.0
✅ GPU aktif: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


In [ ]:
# Upload file kaggle.json
from google.colab import files
print('Silakan upload file kaggle.json kamu:')
uploaded = files.upload()

Silakan upload file kaggle.json kamu:


In [ ]:
import os

# Pindahkan kaggle.json ke lokasi yang benar
os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('/content/kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# Install & download dataset OGYEIv2 (112 kelas obat tablet)
!pip install kaggle -q
!kaggle datasets download -d richardradli/ogyeiv2 --unzip -p /content/dataset

print('\n✅ Dataset berhasil didownload!')

Cek Struktur Dataset

In [ ]:
import os

DATASET_PATH = '/content/dataset'

print('Isi folder dataset:')
for item in sorted(os.listdir(DATASET_PATH)):
    print(' ', item)

# Deteksi subfolder train
# Dataset OGYEIv2 biasanya punya folder: train/ test/ atau langsung per kelas
subfolders = [f for f in os.listdir(DATASET_PATH) if os.path.isdir(os.path.join(DATASET_PATH, f))]
print('\nSubfolder ditemukan:', subfolders)

Observasi dari output `os.listdir(DATASET_PATH)` di atas menunjukkan bahwa ada subfolder bernama 'ogyeiv2' di dalam `/content/dataset`

In [ ]:

if 'train' in subfolders:
    TRAIN_DIR = os.path.join(DATASET_PATH, 'train')
    TEST_DIR  = os.path.join(DATASET_PATH, 'test') if 'test' in subfolders else None
else:
    # Jika langsung berisi folder per kelas
    TRAIN_DIR = DATASET_PATH
    TEST_DIR  = None

# Hitung kelas & gambar
classes = sorted(os.listdir(TRAIN_DIR))
NUM_CLASSES = len(classes)

total_images = sum(
    len(os.listdir(os.path.join(TRAIN_DIR, c)))
    for c in classes
    if os.path.isdir(os.path.join(TRAIN_DIR, c))
)

print(f'Jumlah kelas obat : {NUM_CLASSES}')
print(f'Total gambar      : {total_images}')
print(f'\nContoh nama kelas : {classes[:10]}')

In [ ]:
import os


DATASET_PATH = '/content/dataset'

def find_folder(root_path, target_name):
    for root, dirs, files in os.walk(root_path):
        if target_name in dirs:
            return os.path.join(root, target_name)
    return None


TRAIN_DIR = find_folder(DATASET_PATH, 'train')
TEST_DIR = find_folder(DATASET_PATH, 'test')

# Jika tidak ada folder 'train'
if not TRAIN_DIR:
    TRAIN_DIR = os.path.join(DATASET_PATH, 'ogyeiv2')

# Hitung kelas & gambar
if os.path.exists(TRAIN_DIR):

    classes = [d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))]
    classes.sort()
    NUM_CLASSES = len(classes)

    total_images = sum(
        len(os.listdir(os.path.join(TRAIN_DIR, c)))
        for c in classes
        if os.path.isdir(os.path.join(TRAIN_DIR, c))
    )
else:
    NUM_CLASSES = 0
    total_images = 0
    classes = []

print(f'Detected Train Path: {TRAIN_DIR}')
print(f'Jumlah kelas obat : {NUM_CLASSES}')
print(f'Total gambar      : {total_images}')
print(f'\nContoh nama kelas : {classes[:10]}')

##Import Library

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.applications import MobileNetV2

print('✅ Library siap')

## Persiapan Data & Augmentasi

In [ ]:
IMG_SIZE   = (128, 128)
BATCH_SIZE = 32

# Generator untuk training (dengan augmentasi)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,      # 20% untuk validasi
    rotation_range=20,         # Rotasi acak ±20 derajat
    width_shift_range=0.1,     # Geser horizontal
    height_shift_range=0.1,    # Geser vertikal
    zoom_range=0.15,           # Zoom acak
    horizontal_flip=True,      # Flip kiri-kanan
    brightness_range=[0.8, 1.2],  # Variasi kecerahan
    fill_mode='nearest'
)

# Load data training
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42
)

# Load data validasi (tanpa augmentasi, hanya rescale)
val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42
)

print(f'\nJumlah kelas terdeteksi : {train_generator.num_classes}')
print(f'Data training           : {train_generator.samples} gambar')
print(f'Data validasi           : {val_generator.samples} gambar')

Lihat Contoh Gambar Obat

In [ ]:
# Ambil 1 batch
images, labels = next(train_generator)

# Mapping index → nama kelas
idx_to_class = {v: k for k, v in train_generator.class_indices.items()}

plt.figure(figsize=(14, 6))
for i in range(min(12, len(images))):
    plt.subplot(3, 4, i + 1)
    plt.imshow(images[i])
    nama_kelas = idx_to_class[np.argmax(labels[i])]
    # Tampilkan nama singkat agar muat
    plt.title(nama_kelas[:20], fontsize=7, wrap=True)
    plt.axis('off')

plt.suptitle('Contoh gambar obat tablet dari dataset', fontsize=12)
plt.tight_layout()
plt.show()

## Bangun Model (Transfer Learning MobileNetV2)


In [ ]:
NUM_CLASSES = train_generator.num_classes

# ── Base model: MobileNetV2 (pretrained ImageNet, tanpa top layer) ──
base_model = MobileNetV2(
    input_shape=(128, 128, 3),
    include_top=False,          # Hapus layer classifier aslinya
    weights='imagenet'          # Pakai bobot pretrained
)

# Freeze semua layer base (tidak dilatih dulu)
base_model.trainable = False

# ── Bangun model lengkap ──
model = models.Sequential([
    base_model,

    # Global Average Pooling (ganti Flatten, lebih efisien)
    layers.GlobalAveragePooling2D(),

    # Dense layer pertama
    layers.Dense(512, activation='relu',
                 kernel_regularizer=regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.Dropout(0.5),

    # Dense layer kedua
    layers.Dense(256, activation='relu',
                 kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.3),

    # Output layer — jumlah neuron = jumlah kelas obat
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.summary()

## Compile & Siapkan Callbacks

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    # Simpan model terbaik secara otomatis
    ModelCheckpoint(
        'best_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    # Hentikan training jika tidak ada peningkatan
    EarlyStopping(
        monitor='val_accuracy',
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    # Kurangi learning rate otomatis jika stagnan
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

print('✅ Model siap dilatih!')

## Fase 1: Latih Classifier Layer Saja

Base MobileNetV2 dibekukan dulu. Kita hanya latih layer classifier baru.

In [ ]:
print('=== FASE 1: Melatih classifier layer ===')
print(f'Layer trainable: {sum(1 for l in model.layers if l.trainable)}')

history_phase1 = model.fit(
    train_generator,
    epochs=15,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

print('\n✅ Fase 1 selesai!')

## Fase 2: Fine-Tuning (Unfreeze sebagian base model)

Setelah classifier terlatih, kita buka sebagian layer base MobileNetV2
dan latih lagi dengan learning rate sangat kecil untuk hasil lebih optimal.

In [ ]:
print('=== FASE 2: Fine-tuning ===')

# Unfreeze 30 layer terakhir dari base model
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f'Layer base yang dilatih: {trainable_count} dari {len(base_model.layers)}')

# Compile ulang dengan learning rate sangat kecil
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_phase2 = model.fit(
    train_generator,
    epochs=20,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

print('\n✅ Fase 2 selesai!')

## Visualisasi Hasil Training

In [ ]:
def plot_history(h1, h2):
    # Gabungkan history dua fase
    acc     = h1.history['accuracy']     + h2.history['accuracy']
    val_acc = h1.history['val_accuracy'] + h2.history['val_accuracy']
    loss    = h1.history['loss']         + h2.history['loss']
    val_loss= h1.history['val_loss']     + h2.history['val_loss']
    epochs  = range(1, len(acc) + 1)
    switch  = len(h1.history['accuracy'])  # Titik peralihan fase

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

    for ax, train_data, val_data, title, ylabel in [
        (ax1, acc,  val_acc,  'Akurasi per Epoch', 'Akurasi'),
        (ax2, loss, val_loss, 'Loss per Epoch',    'Loss')
    ]:
        ax.plot(epochs, train_data, label='Training', color='steelblue')
        ax.plot(epochs, val_data,   label='Validasi',  color='coral')
        ax.axvline(x=switch, color='gray', linestyle='--', alpha=0.7, label='Mulai fine-tune')
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_history(history_phase1, history_phase2)

##  Evaluasi Final

In [ ]:
val_loss, val_acc = model.evaluate(val_generator, verbose=0)
print(f'Akurasi validasi : {val_acc * 100:.2f}%')
print(f'Loss validasi    : {val_loss:.4f}')

### Analisis Performa Detail (Confusion Matrix & Report)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

val_generator.reset()
y_pred_probs = model.predict(val_generator, steps=len(val_generator))
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = val_generator.classes
class_labels = list(val_generator.class_indices.keys())

print('Classification Report:')
print(classification_report(y_true, y_pred, target_names=class_labels))

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(20, 18))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues')
plt.title('Confusion Matrix (112 Kelas Obat)')
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.show()

## Prediksi Gambar Obat

In [ ]:
from tensorflow.keras.preprocessing import image as keras_image

# Ambil beberapa gambar dari validasi untuk dicoba
val_generator.reset()
images_batch, labels_batch = next(val_generator)

predictions = model.predict(images_batch[:16], verbose=0)
idx_to_class = {v: k for k, v in val_generator.class_indices.items()}

plt.figure(figsize=(14, 10))
for i in range(16):
    plt.subplot(4, 4, i + 1)
    plt.imshow(images_batch[i])

    pred_idx   = np.argmax(predictions[i])
    true_idx   = np.argmax(labels_batch[i])
    pred_name  = idx_to_class[pred_idx][:18]
    true_name  = idx_to_class[true_idx][:18]
    confidence = predictions[i][pred_idx] * 100
    benar      = pred_idx == true_idx

    plt.title(
        f'Pred: {pred_name}\nAsli: {true_name}\n{confidence:.1f}%',
        fontsize=6.5,
        color='green' if benar else 'red'
    )
    plt.axis('off')

plt.suptitle('Prediksi Jenis Obat Tablet (hijau=benar, merah=salah)', fontsize=12)
plt.tight_layout()
plt.show()

## Prediksi Foto Obat Sendiri

In [ ]:
from google.colab import files
from tensorflow.keras.preprocessing import image as keras_image

print('Upload foto obat tablet kamu (jpg/png):')
uploaded = files.upload()

for filename in uploaded.keys():
    img = keras_image.load_img(filename, target_size=IMG_SIZE)
    img_array = keras_image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    pred = model.predict(img_array, verbose=0)[0]
    top3_idx = np.argsort(pred)[::-1][:3]

    print(f'\n📷 File: {filename}')
    print('🔝 Top-3 Prediksi:')
    for rank, idx in enumerate(top3_idx, 1):
        print(f'  {rank}. {idx_to_class[idx]} — {pred[idx]*100:.1f}%')

    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.title(f'Prediksi: {idx_to_class[top3_idx[0]]}\n({pred[top3_idx[0]]*100:.1f}%)', fontsize=10)
    plt.axis('off')
    plt.show()

##  Simpanke Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

save_path = '/content/drive/MyDrive/cnn_obat_tablet.h5'
model.save(save_path)
print(f'✅ Model tersimpan di Google Drive: {save_path}')